In [84]:
import numpy as np
import torch
import torch.nn.functional as F

In [85]:
### Building a triagram

words = open('names.txt','r').read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}



In [86]:
xs, ys = [], []
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    ix1, ix2, ix3 = stoi[ch1], stoi[ch2], stoi[ch3]
    xs.append([ix1,ix2])
    ys.append(ix3)

xs, ys = torch.tensor(xs), torch.tensor(ys)


In [87]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27*27, 27), generator=g, requires_grad=True)

In [95]:
num, _ = xs.shape
xen = xs @ torch.tensor([27,1])
xenc = F.one_hot(xen, num_classes=27*27).float()
for k in range(30):
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
    loss = -probs[torch.arange(num), ys].log().mean()
    print(loss.item())
    W.grad = None # set to zero the gradient
    loss.backward()
    
    # update
    W.data += -100 * W.grad

2.236898183822632
2.2357370853424072
2.2345917224884033
2.23346209526062
2.2323474884033203
2.231248140335083
2.2301628589630127
2.2290918827056885
2.2280352115631104
2.226992130279541
2.2259626388549805
2.2249462604522705
2.223942756652832
2.222952127456665
2.2219741344451904
2.221008062362671
2.2200541496276855
2.219111919403076
2.218181610107422
2.2172622680664062
2.2163543701171875
2.2154574394226074
2.214571237564087
2.2136952877044678
2.212830066680908
2.21197509765625
2.211129903793335
2.2102949619293213
2.2094695568084717
2.208653450012207


In [96]:
W

tensor([[ 1.5674, -0.2373, -0.0274,  ..., -0.0707,  2.4968,  2.4448],
        [-2.3446,  0.7694,  0.6828,  ..., -1.2288,  0.5880,  0.4569],
        [-0.7080,  2.9428, -0.6162,  ..., -1.1114, -0.5008, -1.4741],
        ...,
        [-0.5241,  0.4013, -0.3320,  ..., -0.5086, -2.0354, -0.1595],
        [ 1.6089,  0.6242, -1.3836,  ..., -1.1750, -0.0176, -1.0889],
        [ 0.3145,  0.9168,  0.4479,  ...,  1.0341,  0.2849,  1.1644]],
       requires_grad=True)

In [ ]:
P = W.exp()
P /= P.sum(dim=1, keepdim=True)
for i in range(10):
    ix = 0
    p = torch.tensor([1/26 for _ in range(26)])
    n=0
    ix0 = torch.multinomial(p,num_samples=1, replacement=True, generator=g).item() + 1
    name = itos[ix0]
    while True:

        prob = P[ix*27 + ix0] #got rid of the one hot encoding
        ix1 = torch.multinomial(prob,num_samples=1, replacement=True, generator=g).item()
        if ix1 == 0:
            break
        ix, ix0 = ix0, ix1
        name = name + itos[ix1]

    print(name) #the model isn't suppose to be better as the data stills small, there's a lot more sparsity in W with trigram

zflcxwzfpin
woltkenn
jai
yars
ederaton
tinny
umdfon
elcsveljalyn
ra
yaejj
